Dataset audit

In [1]:
from re import search

import pandas as pd
import numpy as np
from IPython import display
from jupyter_builder.jupyterlab_semver import compare_identifiers
from sklearn.metrics import mean_absolute_error, mean_squared_error
from src.data import (
load_raw_data,
clean_delivery_data,
create_train_test_split,
)

raw_df = load_raw_data()
raw_df.info()
display(raw_df.head())

print("=================================================================================")
print(raw_df.describe())
print("=================================================================================")
print(raw_df.columns)
print("=================================null================================================")
print(raw_df.isnull().sum())
print("=================================nan================================================")
print(raw_df.isna().sum())
print("===================================dup============================================")
print(raw_df.duplicated().sum())
print("===================================shape============================================")
print(raw_df.shape)

<class 'pandas.DataFrame'>
RangeIndex: 45593 entries, 0 to 45592
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ID                           45593 non-null  str    
 1   Delivery_person_ID           45593 non-null  str    
 2   Delivery_person_Age          45593 non-null  str    
 3   Delivery_person_Ratings      45593 non-null  str    
 4   Restaurant_latitude          45593 non-null  float64
 5   Restaurant_longitude         45593 non-null  float64
 6   Delivery_location_latitude   45593 non-null  float64
 7   Delivery_location_longitude  45593 non-null  float64
 8   Order_Date                   45593 non-null  str    
 9   Time_Orderd                  45593 non-null  str    
 10  Time_Order_picked            45593 non-null  str    
 11  Weatherconditions            45593 non-null  str    
 12  Road_traffic_density         45593 non-null  str    
 13  Vehicle_condition          

TypeError: 'module' object is not callable

identify the target

the target is the delivery time in minutes its the most important feature to predict , we can rely on it for prediction of the delivery time , its already integer and have a good name that indicate that the values are by minute .

In [ ]:
target = raw_df['Time_taken(min)']
print(target.head(3))
print(target.dtype)

data_dictionary = pd.DataFrame([
    ("ID", "Order identifier", "None", "Exclude: identifier"),
    ("Delivery_person_ID", "Courier identifier", "None", "Exclude: identifier"),
    ("Delivery_person_Age", "Courier age", "Years", "Predictor"),
    ("Delivery_person_Ratings", "Courier rating", "Score", "Predictor"),
    ("Restaurant_latitude", "Restaurant latitude", "Degrees", "Derive distance"),
    ("Restaurant_longitude", "Restaurant longitude", "Degrees", "Derive distance"),
    ("Delivery_location_latitude", "Customer latitude", "Degrees", "Derive distance"),
    ("Delivery_location_longitude", "Customer longitude", "Degrees", "Derive distance"),
    ("Order_Date", "Order date", "Date", "Derive date features"),
    ("Time_Orderd", "Order placement time", "Clock time", "Derive time features"),
    ("Time_Order_picked", "Courier pickup time", "Clock time", "Check availability/leakage"),
    ("Weatherconditions", "Weather condition", "Category", "Predictor"),
    ("Road_traffic_density", "Traffic level", "Category", "Predictor"),
    ("Vehicle_condition", "Vehicle condition level", "Ordinal", "Predictor"),
    ("Type_of_order", "Order category", "Category", "Predictor"),
    ("Type_of_vehicle", "Courier vehicle type", "Category", "Predictor"),
    ("multiple_deliveries", "Concurrent delivery count", "Count", "Predictor"),
    ("Festival", "Festival-day indicator", "Category", "Predictor"),
    ("City", "City category", "Category", "Predictor"),
    ("Time_taken(min)", "Observed delivery duration", "Minutes", "Target"),
], columns=["column", "meaning", "unit", "modeling_decision"])
data_dictionary.insert(2, "dtype", data_dictionary["column"].map(raw_df.dtypes.astype(str)))

display(data_dictionary)

cleaning data setup

In [ ]:
cleaned_df = clean_delivery_data(raw_df)

print("raw shape:", raw_df.shape)
print("cleaned shape:", cleaned_df.shape)
display(cleaned_df.head(3))
print(cleaned_df.isna().sum())
print("exact duplicates", raw_df.duplicated().sum())



Visualisation

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


#time takeny

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(
      cleaned_df["Time_Order_picked"].dt.total_seconds()/3600 ,
      bins=range(25),
      edgecolor="black"
  )
ax.set_title("time taken (hour) vs orders")
ax.set_xticks(range(0, 25, 2))
plt.show()
print(cleaned_df.shape)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.boxplot(x=cleaned_df["Time_taken(min)"])

plt.title("Delivery time — training data")
plt.xlabel("Delivery time (minutes)")
plt.show()

analyse descriptive

In [ ]:

sns.boxplot(cleaned_df,y="Time_taken(min)", x="Weatherconditions")
plt.title("delivery time and weather conditions")
plt.show()


i think sunny has some outliers but if the distance between the restaurant and the delivery location is too far, it may take longer to deliver the food. this could be due to traffic or other factors that affect delivery time. it would be interesting to see if there is a correlation between traffic and delivery time.

In [ ]:
sns.barplot(
    data=cleaned_df,
    x="Road_traffic_density",
    y="Time_taken(min)",
)
plt.xticks([0,1,2,3], ["low", "medium", "high", "jam"])
plt.show()

In [ ]:
sns.boxplot(
    data=cleaned_df,
    x="Type_of_vehicle",
    y="Time_taken(min)",
)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

sns.boxplot(
    data=cleaned_df,
    x="Road_traffic_density",
    y="Time_taken(min)",
    ax=ax,
)

ax.set_title("Delivery time by traffic level")
ax.set_xlabel("Traffic category")
ax.set_ylabel("Delivery time (minutes)")

plt.tight_layout()
plt.show()

correlation matrice

In [ ]:
corr = cleaned_df.loc[:,["Time_taken(min)", "Road_traffic_density"]].corr()
print(cleaned_df.loc[:,["Time_taken(min)", "Road_traffic_density"]].corr())

visualize correlation


In [ ]:
sns.heatmap(corr, annot=True, cmap="coolwarm")

Identifier les variables les plus fortement liées au temps de livraison.

Road traffic and weather conditions

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plot_df = cleaned_df.copy()

# Extract whole pickup hours from timedelta values.
plot_df["pickup_hour"] = (
    (plot_df["Time_Order_picked"].dt.total_seconds() // 3600) % 24
).astype("Int64")

fig, ax = plt.subplots(figsize=(10, 5))

sns.lineplot(
    data=plot_df,
    x="pickup_hour",
    y="Time_taken(min)",
    estimator="mean",
    errorbar=None,
    marker="x",
    ax=ax,
)



ax.set_title("Average delivery time by pickup hour")
ax.set_xlabel("Pickup hour of day")
ax.set_ylabel("Average delivery time (minutes)")
ax.set_xticks(range(25))
plt.tight_layout()
plt.show()

add  two features


In [ ]:
from src.features import create_features
df_features = create_features(cleaned_df)
display(df_features.head())

Training Data

In [5]:
from src.data import *
from src.train import *
from src.features import *
import pandas as pd

featured = create_features(cleaned_df)

print(featured[
  [
      "distance_km",
      "Road_traffic_density",
      "order_hour",
      "Weatherconditions",
      "Type_of_vehicle",
      "Time_taken(min)",
  ]
].head())

X_train, X_test, y_train, y_test = create_train_test_split(featured)
print("xtrain:", X_train.shape)
print("xtest:", X_test.shape)
print("ytrain:", y_train.shape)
print("ytest:", y_test.shape)



cv_results = cross_validate_pipelines(X_train, y_train)
cv_df = pd.DataFrame(cv_results).T
cv_df = cv_df.sort_values("validation_mae")

   distance_km Road_traffic_density  order_hour Weatherconditions  \
0     3.025153                    2          11             Sunny   
1    20.183558                    3          19            Stormy   
2     1.552760                    0           8        Sandstorms   
3     7.790412                    1          18             Sunny   
4     6.210147                    2          13            Cloudy   

  Type_of_vehicle  Time_taken(min)  
0      Motorcycle               24  
1         Scooter               33  
2      Motorcycle               26  
3      Motorcycle               21  
4         Scooter               30  
xtrain: (33562, 11)
xtest: (8391, 11)
ytrain: (33562,)
ytest: (8391,)


Optimazing

In [13]:
from IPython.display import display
print(cv_df.dtypes)
display(cv_df)

validation_mae     float64
validation_mse     float64
validation_rmse    float64
validation_r2      float64
training_mae       float64
fit_time           float64
dtype: object


,validation_mae,validation_mse,validation_rmse,validation_r2,training_mae,fit_time
random_forest,5.852910,57.249205,7.566131,0.352729,2.307132,0.694891
ridge_regression,6.255075,60.013364,7.746591,0.321492,6.252341,0.048571
linear_regression,6.255080,60.013542,7.746603,0.321490,6.252333,0.062422
decision_tree,7.195223,88.704614,9.417996,-0.003060,0.360182,0.104983
dummy_baseline,7.584291,88.560439,9.410452,-0.001214,7.584292,0.066739


In [14]:
from src.tuning import *

random_forest_tune = tune_model("random_forest", X_train, y_train)


Fitting 5 folds for each of 15 candidates, totalling 75 fits


In [18]:
test = random_forest_tune.best_estimator_.predict(X_test)
comparison_df = pd.DataFrame({
    "actual_value": y_test,
    "predicted_value": test
})

comparison_df["error"] = comparison_df["actual_value"] - comparison_df["predicted_value"]
comparison_df["absolute_error"] = comparison_df["error"].abs()
display(comparison_df.head(3))

,actual_value,predicted_value,error,absolute_error
20657,21,21.838054,-0.838054,0.838054
17666,18,20.354170,-2.354170,2.354170
4615,25,35.047063,-10.047063,10.047063


In [35]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error

#before randomsearch
display(cv_df.loc[["random_forest"]])

#after tunning on training

result_after_tuning = random_forest_tune.cv_results_
bestCombination = random_forest_tune.best_index_

best_cv_metrics = {
    "mae": -result_after_tuning[
        "mean_test_mae"
    ][bestCombination],

    "mse": -result_after_tuning[
        "mean_test_mse"
    ][bestCombination],

    "rmse": -result_after_tuning[
        "mean_test_rmse"
    ][bestCombination],

    "r2": result_after_tuning[
        "mean_test_r2"
    ][bestCombination],
}

display(pd.DataFrame([best_cv_metrics]))

#after tunning with test
print("after tunning with test")
print("mae",mean_absolute_error(y_test, test))
print("mse",mean_squared_error(y_test, test))
print("r2",r2_score(y_test, test))
print("rmse",root_mean_squared_error(y_test, test))


,validation_mae,validation_mse,validation_rmse,validation_r2,training_mae,fit_time
random_forest,5.85291,57.249205,7.566131,0.352729,2.307132,0.694891


,mae,mse,rmse,r2
0,5.233068,44.515028,6.67152,0.496777


after tunning with test
mae 5.177764125135159
mse 43.252441747977976
r2 0.4979758573666012
rmse 6.576658859023933
